In [1]:
from dotenv import load_dotenv

# Load API KEY information
load_dotenv(override=True)

True

## VectorStoreRetriever

In [2]:
from langchain_community.vectorstores import FAISS
from langchain_openai.embeddings import OpenAIEmbeddings
from langchain_text_splitters import CharacterTextSplitter
from langchain_community.document_loaders import TextLoader

In [4]:
loader = TextLoader("01-vectorstore-retriever-appendix-keywords.txt", encoding="utf-8")
documents = loader.load()

text_splitter = CharacterTextSplitter(chunk_size=300, chunk_overlap=0)
split_docs = text_splitter.split_documents(documents)

embeddings = OpenAIEmbeddings()

db = FAISS.from_documents(split_docs, embeddings)

Created a chunk of size 351, which is longer than the specified 300
Created a chunk of size 343, which is longer than the specified 300
Created a chunk of size 307, which is longer than the specified 300
Created a chunk of size 316, which is longer than the specified 300
Created a chunk of size 341, which is longer than the specified 300
Created a chunk of size 321, which is longer than the specified 300
Created a chunk of size 303, which is longer than the specified 300
Created a chunk of size 325, which is longer than the specified 300
Created a chunk of size 315, which is longer than the specified 300
Created a chunk of size 304, which is longer than the specified 300
Created a chunk of size 385, which is longer than the specified 300
Created a chunk of size 349, which is longer than the specified 300
Created a chunk of size 376, which is longer than the specified 300


In [5]:
retriever = db.as_retriever()

In [6]:
retriever = db.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={
        "k": 5,
        "score_threshold": 0.7
    }
)

query = "Explain the concept of vector search."
results = retriever.invoke(query)

for doc in results:
    print(doc.page_content)


Semantic Search
VectorStore

Definition: A vector store is a system for storing data in vector format, often used for search, classification, and data analysis tasks.
Example: Storing word embeddings in a database for fast retrieval of similar words.
Related Keywords: Embedding, Database, Vectorization
Definition: Semantic search is a method of retrieving results based on the meaning of the user's query, going beyond simple keyword matching.
Example: If a user searches for "solar system planets," the search returns information about related planets like Jupiter and Mars.
Related Keywords: Natural Language Processing, Search Algorithms, Data Mining
Definition: Keyword search is the process of finding information based on specific keywords entered by the user. It is commonly used in search engines and database systems as a fundamental search method.
Example: If a user searches for "coffee shop in Seoul," the search engine returns a list of related coffee shops.
Related Keywords: Search E

In [7]:
from langchain_core.runnables.config import RunnableConfig

config = RunnableConfig(
    tags=["retriever", "faq"],
    metadata={"project": "vectorstore-tutorial"}
)

docs = retriever.invoke(
    input="What is a DataFrame?",
    config=config,
    search_kwargs={
        "k": 3,
        "score_threshold": 0.8
    }
)

#  Display the search results
for idx, doc in enumerate(docs):
    print(f"\n🔍 [Search Result {idx + 1}]")
    print("📄 Document Content:", doc.page_content)
    print("🗂️ Metadata:", doc.metadata)
    print("=" * 60)


🔍 [Search Result 1]
📄 Document Content: Definition: A DataFrame is a tabular data structure with rows and columns, commonly used for data analysis and manipulation.
Example: Pandas DataFrame can store data like an Excel sheet and perform operations like filtering and grouping.
Related Keywords: Data Analysis, Pandas, Data Manipulation
🗂️ Metadata: {'source': '01-vectorstore-retriever-appendix-keywords.txt'}

🔍 [Search Result 2]
📄 Document Content: Schema

Definition: A schema defines the structure of a database or file, describing how data is stored and organized.
Example: A database schema can specify table columns, data types, and constraints.
Related Keywords: Database, Data Modeling, Data Management

DataFrame
🗂️ Metadata: {'source': '01-vectorstore-retriever-appendix-keywords.txt'}

🔍 [Search Result 3]
📄 Document Content: Pandas

Definition: Pandas is a Python library for data analysis and manipulation, offering tools for working with structured data.
Example: Pandas can read CSV

In [8]:
# MMR Retriever Configuration (Balancing Relevance and Diversity)
retriever = db.as_retriever(
    search_type="mmr", 
    search_kwargs={
        "k": 3,                
        "fetch_k": 10,           
        "lambda_mult": 0.6  # Balancing Similarity and Diversity (0.6: Slight Emphasis on Diversity)
    }
)

query = "What is an embedding?"
docs = retriever.invoke(query)

#  Display the search results
print(f"\n🔎 [Query]: {query}\n")
for idx, doc in enumerate(docs):
    print(f"📄 [Document {idx + 1}]")
    print("📖 Document Content:", doc.page_content)
    print("🗂️ Metadata:", doc.metadata)
    print("=" * 60)


🔎 [Query]: What is an embedding?

📄 [Document 1]
📖 Document Content: Embedding
🗂️ Metadata: {'source': '01-vectorstore-retriever-appendix-keywords.txt'}
📄 [Document 2]
📖 Document Content: Definition: Embedding is the process of converting text data, such as words or sentences, into continuous low-dimensional vectors. This allows computers to understand and process text.
Example: The word "apple" can be represented as a vector like [0.65, -0.23, 0.17].
Related Keywords: Natural Language Processing, Vectorization, Deep Learning
🗂️ Metadata: {'source': '01-vectorstore-retriever-appendix-keywords.txt'}
📄 [Document 3]
📖 Document Content: TF-IDF (Term Frequency-Inverse Document Frequency)
🗂️ Metadata: {'source': '01-vectorstore-retriever-appendix-keywords.txt'}


In [9]:
from langchain_core.runnables import ConfigurableField

retriever = db.as_retriever(search_kwargs={"k": 1}).configurable_fields(
    search_type=ConfigurableField(
        id="search_type",
        name="Search Type",
        description="The search type to use",
    ),
    search_kwargs=ConfigurableField(
        id="search_kwargs",
        name="Search Kwargs",
        description="The search kwargs to use",
    ),
)

In [10]:
# ✅ Search Configuration 3: MMR Search (Diversity and Relevance Balanced)

config_3 = {
    "configurable": {
        "search_type": "mmr",
        "search_kwargs": {
            "k": 2,            # Return the top 2 most diverse and relevant documents
            "fetch_k": 10,     # Initially fetch the top 10 documents before filtering for diversity
            "lambda_mult": 0.6 # Balance factor: 0.6 (0 = maximum diversity, 1 = maximum relevance)
        },
    }
}
# Execute the query using MMR search
docs = retriever.invoke("What is Word2Vec?", config=config_3)

#  Display the search results
print("\n🔎 [Search Results - MMR (Diversity and Relevance Balanced)]")
for idx, doc in enumerate(docs):
    print(f"📄 [Document {idx + 1}]")
    print(doc.page_content)
    print("=" * 60)


🔎 [Search Results - MMR (Diversity and Relevance Balanced)]
📄 [Document 1]
Word2Vec

Definition: Word2Vec is a technique in NLP that maps words into a vector space, representing their semantic relationships based on context.
Example: In Word2Vec, "king" and "queen" would be represented by vectors close to each other.
Related Keywords: NLP, Embeddings, Semantic Similarity
📄 [Document 2]
Tokenizer


## ContextualCompressional

In [11]:
# Helper function to print documents in a pretty format
def pretty_print_docs(docs):
    print(
        f"\n{'-' * 100}\n".join(
            [f"document {i+1}:\n\n" + d.page_content for i, d in enumerate(docs)]
        )
    )

In [18]:
from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import CharacterTextSplitter

# 1. Generate Loader to lthe text file using TextLoader
loader = TextLoader("appendix-keywords.txt", encoding="utf-8")
documents = loader.load()

# 2. Generate text chunks using CharacterTextSplitter and split the text into chunks of 300 characters with no overlap.
text_splitter = CharacterTextSplitter(chunk_size=300, chunk_overlap=0)
texts = text_splitter.split_documents(documents)

# 3. Generate vector store using FAISS and convert it to retriever
retriever = FAISS.from_documents(texts, OpenAIEmbeddings()).as_retriever()

# 4. Query the retriever to find relevant documents
docs = retriever.invoke("What is the definition of Multimodal?")

# 5. Print the relevant documents
pretty_print_docs(docs)

Created a chunk of size 392, which is longer than the specified 300
Created a chunk of size 359, which is longer than the specified 300
Created a chunk of size 313, which is longer than the specified 300
Created a chunk of size 325, which is longer than the specified 300
Created a chunk of size 348, which is longer than the specified 300
Created a chunk of size 301, which is longer than the specified 300
Created a chunk of size 363, which is longer than the specified 300
Created a chunk of size 357, which is longer than the specified 300
Created a chunk of size 384, which is longer than the specified 300
Created a chunk of size 356, which is longer than the specified 300
Created a chunk of size 318, which is longer than the specified 300
Created a chunk of size 324, which is longer than the specified 300
Created a chunk of size 301, which is longer than the specified 300
Created a chunk of size 354, which is longer than the specified 300
Created a chunk of size 364, which is longer tha

document 1:

Multimodal
----------------------------------------------------------------------------------------------------
document 2:

Definition: Multimodal refers to combining multiple types of data (e.g., text, images, audio) for processing. It is used to extract or predict richer and more accurate information through cross-modal interactions.
Example: A system that analyzes images and descriptive text together for better image classification is an example of multimodal technology.
Related Keywords: Data Fusion, Artificial Intelligence, Deep Learning
----------------------------------------------------------------------------------------------------
document 3:

Semantic Search
----------------------------------------------------------------------------------------------------
document 4:

LLM (Large Language Model)


In [19]:
from langchain.retrievers import ContextualCompressionRetriever
from langchain.retrievers.document_compressors import LLMChainExtractor
from langchain_openai import ChatOpenAI

# Before applying ContextualCompressionRetriever
pretty_print_docs(retriever.invoke("What is the definition of Multimodal?"))
print("="*62)
print("="*15 + "After applying LLMChainExtractor" + "="*15)

llm = ChatOpenAI(temperature=0, model="gpt-4o-mini")

compressor = LLMChainExtractor.from_llm(llm)

compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=retriever,
)

compressed_docs = (
    compression_retriever.invoke(
        "What is the definition of Multimodal?"
    )
)

pretty_print_docs(compressed_docs)

document 1:

Multimodal
----------------------------------------------------------------------------------------------------
document 2:

Definition: Multimodal refers to combining multiple types of data (e.g., text, images, audio) for processing. It is used to extract or predict richer and more accurate information through cross-modal interactions.
Example: A system that analyzes images and descriptive text together for better image classification is an example of multimodal technology.
Related Keywords: Data Fusion, Artificial Intelligence, Deep Learning
----------------------------------------------------------------------------------------------------
document 3:

Semantic Search
----------------------------------------------------------------------------------------------------
document 4:

LLM (Large Language Model)
===============After applying LLMChainExtractor===============
document 1:

Multimodal
-------------------------------------------------------------------------------

In [20]:
from langchain.retrievers.document_compressors import LLMChainFilter
filter = LLMChainFilter.from_llm(llm)

compression_retriever = ContextualCompressionRetriever(
    base_compressor=filter,
    base_retriever=retriever,
)

compressed_docs = compression_retriever.invoke(
    "What is the definition of Multimodal"
)

pretty_print_docs(compressed_docs)

document 1:

Multimodal
----------------------------------------------------------------------------------------------------
document 2:

Definition: Multimodal refers to combining multiple types of data (e.g., text, images, audio) for processing. It is used to extract or predict richer and more accurate information through cross-modal interactions.
Example: A system that analyzes images and descriptive text together for better image classification is an example of multimodal technology.
Related Keywords: Data Fusion, Artificial Intelligence, Deep Learning


In [21]:
from langchain.retrievers.document_compressors import EmbeddingsFilter
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings()

embeddings_filter = EmbeddingsFilter(embeddings=embeddings, similarity_threshold=0.86)

compression_retriever = ContextualCompressionRetriever(
    base_compressor=embeddings_filter,
    base_retriever=retriever
)

# 4. Query the compression retriever to find relevant documents
compressed_docs = compression_retriever.invoke(
    "What is the definition of Multimodal?"
)

# 5. Print the relevant documents
pretty_print_docs(compressed_docs)

document 1:

Multimodal
----------------------------------------------------------------------------------------------------
document 2:

Definition: Multimodal refers to combining multiple types of data (e.g., text, images, audio) for processing. It is used to extract or predict richer and more accurate information through cross-modal interactions.
Example: A system that analyzes images and descriptive text together for better image classification is an example of multimodal technology.
Related Keywords: Data Fusion, Artificial Intelligence, Deep Learning


In [22]:
from langchain.retrievers.document_compressors import DocumentCompressorPipeline
from langchain_community.document_transformers import EmbeddingsRedundantFilter
from langchain_text_splitters import CharacterTextSplitter

splitter = CharacterTextSplitter(chunk_size=300, chunk_overlap=0)

redundant_filter = EmbeddingsRedundantFilter(embeddings=embeddings)

relevant_filter = EmbeddingsFilter(embeddings=embeddings, similarity_threshold=0.86)

pipeline_compressor = DocumentCompressorPipeline(
    transformers=[
        splitter,
        redundant_filter,
        relevant_filter,
        LLMChainExtractor.from_llm(llm)
    ]
)

In [23]:
compression_retriever = ContextualCompressionRetriever(
    base_compressor=pipeline_compressor,
    base_retriever=retriever,
)

compressed_docs = compression_retriever.invoke(
    "What is the definition of Multimodal?"
)

pretty_print_docs(compressed_docs)

document 1:

Multimodal
----------------------------------------------------------------------------------------------------
document 2:

Definition: Multimodal refers to combining multiple types of data (e.g., text, images, audio) for processing. It is used to extract or predict richer and more accurate information through cross-modal interactions.


## EnsembleRetriever

In [2]:
from langchain.retrievers import BM25Retriever, EnsembleRetriever
from langchain.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings

# list sample documents
doc_list = [
    "I like apples",
    "I like apple company",
    "I like apple's iphone",
    "Apple is my favorite company",
    "I like apple's ipad",
    "I like apple's macbook",
]

bm25_retriever = BM25Retriever.from_texts(doc_list)

bm25_retriever.k = 1

embeddings = OpenAIEmbeddings()

faiss_vectorstore = FAISS.from_texts(doc_list, embeddings)

faiss_retriever = faiss_vectorstore.as_retriever(search_kwargs={"k": 1})

ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, faiss_retriever],
    weights=[0.3, 0.7]
)


In [3]:
query = "my favorite fruit is apple"
ensemble_result = ensemble_retriever.invoke(query)
bm25_result = bm25_retriever.invoke(query)
faiss_result = faiss_retriever.invoke(query)

# Output the fetched documents.
print("[Ensemble Retriever]")
for doc in ensemble_result:
    print(f"Content: {doc.page_content}")
    print()

print("[BM25 Retriever]")
for doc in bm25_result:
    print(f"Content: {doc.page_content}")
    print()

print("[FAISS Retriever]")
for doc in faiss_result:
    print(f"Content: {doc.page_content}")
    print()

[Ensemble Retriever]
Content: I like apples

Content: Apple is my favorite company

[BM25 Retriever]
Content: Apple is my favorite company

[FAISS Retriever]
Content: I like apples



In [4]:
from langchain_core.runnables import ConfigurableField

ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, faiss_retriever],
).configurable_fields(
    weights=ConfigurableField(
        id="ensemble_weights",
        name="Ensemble Weights",
        description="Ensemble Weights"
    )
)

In [5]:
config = {"configurable": {"ensemble_weights": [1, 0]}}

docs = ensemble_retriever.invoke("my favorite fruit is apple", config=config)
docs


[Document(metadata={}, page_content='Apple is my favorite company'),
 Document(id='bec95c61-e0c4-43bc-addc-8672c087b85d', metadata={}, page_content='I like apples')]

In [6]:
config = {"configurable": {"ensemble_weights": [0.1, 0.9]}}

# Use the config parameter to specify search settings.
docs = ensemble_retriever.invoke("my favorite fruit is apple", config=config)
docs  # Print the search result, docs.

[Document(id='bec95c61-e0c4-43bc-addc-8672c087b85d', metadata={}, page_content='I like apples'),
 Document(metadata={}, page_content='Apple is my favorite company')]

## LongContextReorder

In [11]:
from langchain_core.prompts import PromptTemplate
from langchain_community.document_transformers import LongContextReorder
from langchain_community.vectorstores import Chroma
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

texts = [
    "This is just a random text I wrote.",
    "ChatGPT, an AI designed to converse with users, can answer various questions.",
    "iPhone, iPad, MacBook are representative products released by Apple.",
    "ChatGPT was developed by OpenAI and is continuously being improved.",
    "ChatGPT has learned from vast amounts of data to understand user questions and generate appropriate answers.",
    "Wearable devices like Apple Watch and AirPods are also part of Apple's popular product line.",
    "ChatGPT can be used to solve complex problems or suggest creative ideas.",
    "Bitcoin is also called digital gold and is gaining popularity as a store of value.",
    "ChatGPT's capabilities are continuously evolving through ongoing learning and updates.",
    "The FIFA World Cup is held every four years and is the biggest event in international football.",
]

retriever = Chroma.from_texts(texts, embeddings).as_retriever(
    search_kwargs={"k": 10}
)

In [12]:
query = "What can you tell me about ChatGPT?"

# Retrieves relevant documents sorted by relevance score.
docs = retriever.invoke(query)
docs

[Document(metadata={}, page_content='ChatGPT was developed by OpenAI and is continuously being improved.'),
 Document(metadata={}, page_content='ChatGPT was developed by OpenAI and is continuously being improved.'),
 Document(metadata={}, page_content='ChatGPT was developed by OpenAI and is continuously being improved.'),
 Document(metadata={}, page_content='ChatGPT, an AI designed to converse with users, can answer various questions.'),
 Document(metadata={}, page_content='ChatGPT, an AI designed to converse with users, can answer various questions.'),
 Document(metadata={}, page_content='ChatGPT, an AI designed to converse with users, can answer various questions.'),
 Document(metadata={}, page_content='ChatGPT can be used to solve complex problems or suggest creative ideas.'),
 Document(metadata={}, page_content='ChatGPT can be used to solve complex problems or suggest creative ideas.'),
 Document(metadata={}, page_content='ChatGPT can be used to solve complex problems or suggest cr

In [14]:
reordering = LongContextReorder()
reordered_docs = reordering.transform_documents(docs)
reordered_docs

[Document(metadata={}, page_content='ChatGPT was developed by OpenAI and is continuously being improved.'),
 Document(metadata={}, page_content='ChatGPT, an AI designed to converse with users, can answer various questions.'),
 Document(metadata={}, page_content='ChatGPT, an AI designed to converse with users, can answer various questions.'),
 Document(metadata={}, page_content='ChatGPT can be used to solve complex problems or suggest creative ideas.'),
 Document(metadata={}, page_content='ChatGPT has learned from vast amounts of data to understand user questions and generate appropriate answers.'),
 Document(metadata={}, page_content='ChatGPT can be used to solve complex problems or suggest creative ideas.'),
 Document(metadata={}, page_content='ChatGPT can be used to solve complex problems or suggest creative ideas.'),
 Document(metadata={}, page_content='ChatGPT, an AI designed to converse with users, can answer various questions.'),
 Document(metadata={}, page_content='ChatGPT was d

In [17]:
def format_docs(docs):
    return "\n".join([doc.page_content for i, doc in enumerate(docs)])

In [18]:
print(format_docs(docs))

ChatGPT was developed by OpenAI and is continuously being improved.
ChatGPT was developed by OpenAI and is continuously being improved.
ChatGPT was developed by OpenAI and is continuously being improved.
ChatGPT, an AI designed to converse with users, can answer various questions.
ChatGPT, an AI designed to converse with users, can answer various questions.
ChatGPT, an AI designed to converse with users, can answer various questions.
ChatGPT can be used to solve complex problems or suggest creative ideas.
ChatGPT can be used to solve complex problems or suggest creative ideas.
ChatGPT can be used to solve complex problems or suggest creative ideas.
ChatGPT has learned from vast amounts of data to understand user questions and generate appropriate answers.


In [19]:
def format_docs(docs):
    return "\n".join(
        [
            f"[{i}] {doc.page_content} [source: teddylee777@gmail.com]"
            for i, doc in enumerate(docs)
        ]
    )

def reorder_documents(docs):
    reordering = LongContextReorder()
    reordered_docs = reordering.transform_documents(docs)
    combined = format_docs(reordered_docs)
    print(combined)
    return combined

In [20]:
_ = reorder_documents(docs)

[0] ChatGPT was developed by OpenAI and is continuously being improved. [source: teddylee777@gmail.com]
[1] ChatGPT, an AI designed to converse with users, can answer various questions. [source: teddylee777@gmail.com]
[2] ChatGPT, an AI designed to converse with users, can answer various questions. [source: teddylee777@gmail.com]
[3] ChatGPT can be used to solve complex problems or suggest creative ideas. [source: teddylee777@gmail.com]
[4] ChatGPT has learned from vast amounts of data to understand user questions and generate appropriate answers. [source: teddylee777@gmail.com]
[5] ChatGPT can be used to solve complex problems or suggest creative ideas. [source: teddylee777@gmail.com]
[6] ChatGPT can be used to solve complex problems or suggest creative ideas. [source: teddylee777@gmail.com]
[7] ChatGPT, an AI designed to converse with users, can answer various questions. [source: teddylee777@gmail.com]
[8] ChatGPT was developed by OpenAI and is continuously being improved. [source: t

In [23]:
from langchain.prompts import ChatPromptTemplate
from operator import itemgetter
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda

# Define prompt template
template = """Given this text extracts:
{context}

-----
Please answer the following question:
{question}

Answer in the following languages: {language}
"""

prompt = ChatPromptTemplate.from_template(template)

chain = (
    {
        "context": itemgetter("question")
        | retriever
        | RunnableLambda(reorder_documents),
        "question": itemgetter("question"),
        "language": itemgetter("language")
    }
    | prompt
    | ChatOpenAI(model="gpt-4o-mini")
    | StrOutputParser()

)

In [24]:
answer = chain.invoke(
    {"question": "What can you tell me about ChatGPT?", "language": "English"}
)

[0] ChatGPT was developed by OpenAI and is continuously being improved. [source: teddylee777@gmail.com]
[1] ChatGPT, an AI designed to converse with users, can answer various questions. [source: teddylee777@gmail.com]
[2] ChatGPT, an AI designed to converse with users, can answer various questions. [source: teddylee777@gmail.com]
[3] ChatGPT can be used to solve complex problems or suggest creative ideas. [source: teddylee777@gmail.com]
[4] ChatGPT has learned from vast amounts of data to understand user questions and generate appropriate answers. [source: teddylee777@gmail.com]
[5] ChatGPT can be used to solve complex problems or suggest creative ideas. [source: teddylee777@gmail.com]
[6] ChatGPT can be used to solve complex problems or suggest creative ideas. [source: teddylee777@gmail.com]
[7] ChatGPT, an AI designed to converse with users, can answer various questions. [source: teddylee777@gmail.com]
[8] ChatGPT was developed by OpenAI and is continuously being improved. [source: t

In [25]:
print(answer)

ChatGPT is an AI developed by OpenAI that is designed to engage in conversation with users. It is capable of answering a wide variety of questions and can assist in solving complex problems or generating creative ideas. The AI has been trained on vast amounts of data, allowing it to understand user inquiries and provide appropriate responses. Additionally, ChatGPT is continuously being improved to enhance its capabilities and performance.


## ParentDocumentRetriever

In [27]:
from langchain.storage import InMemoryStore
from langchain_community.document_loaders import TextLoader
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain.retrievers import ParentDocumentRetriever

In [28]:
loaders = [TextLoader("appendix-keywords.txt", encoding="utf-8")]

docs = []
for loader in loaders:
    docs.extend(loader.load())

In [29]:
docs

[Document(metadata={'source': 'appendix-keywords.txt'}, page_content='Semantic Search\n\nDefinition: Semantic search is a search method that goes beyond simple keyword matching by understanding the meaning of the user’s query to return relevant results.\nExample: If a user searches for “planets in the solar system,” the system might return information about related planets such as “Jupiter” or “Mars.”\nRelated Keywords: Natural Language Processing, Search Algorithms, Data Mining\n\nEmbedding\n\nDefinition: Embedding is the process of converting textual data, such as words or sentences, into low-dimensional continuous vectors. This allows computers to better understand and process the text.\nExample: The word “apple” might be represented as a vector like [0.65, -0.23, 0.17].\nRelated Keywords: Natural Language Processing, Vectorization, Deep Learning\n\nToken\n\nDefinition: A token refers to a smaller unit of text obtained by splitting a larger text. It can be a word, sentence, or phras

In [30]:
child_splitter = RecursiveCharacterTextSplitter(chunk_size=200)

vectorstore = Chroma(
    collection_name="full_documents", embedding_function=OpenAIEmbeddings()
)

store = InMemoryStore()

retriever = ParentDocumentRetriever(
    vectorstore=vectorstore,
    docstore=store,
    child_splitter=child_splitter,

)

In [31]:
retriever.add_documents(docs, ids=None, add_to_docstore=True)

In [32]:
list(store.yield_keys())

['5d9f621d-612c-4b1a-8886-50f156f36d63']

In [33]:
sub_docs = vectorstore.similarity_search("Word2Vec")
print(sub_docs[0].page_content)

Word2Vec


In [36]:
retrieved_docs = retriever.invoke("Word2Vec")

In [37]:
# Print the length of the page content of the retrieved document
print(
    f"Document length: {len(retrieved_docs[0].page_content)}",
    end="\n\n=====================\n\n",
)

# Print a portion of the document
print(retrieved_docs[0].page_content[2000:2500])

Document length: 10650


Database, Query, Data Management

CSV

Definition: CSV (Comma-Separated Values) is a file format for storing data where each value is separated by a comma. It is often used for simple data storage and exchange in tabular form.
Example: A CSV file with headers “Name, Age, Job” might contain data like “John Doe, 30, Developer”.
Related Keywords: File Format, Data Handling, Data Exchange

JSON

Definition: JSON (JavaScript Object Notation) is a lightweight data exchange format that represents data 


In [38]:
# Text splitter used to generate parent documents
parent_splitter = RecursiveCharacterTextSplitter(chunk_size=1000)

# Text splitter used to generate child documents
# Should create documents smaller than the parent
child_splitter = RecursiveCharacterTextSplitter(chunk_size=200)

# Vector store to be used for indexing child chunks
vectorstore = Chroma(
    collection_name="split_parents", embedding_function=OpenAIEmbeddings()
)
# Storage layer for parent documents
store = InMemoryStore()

In [39]:
retriever = ParentDocumentRetriever(
    vectorstore=vectorstore,
    docstore=store,
    child_splitter=child_splitter,
    parent_splitter=parent_splitter
)

In [40]:
retriever.add_documents(docs)

In [41]:
len(list(store.yield_keys()))

13

In [42]:
# Perform similarity search
sub_docs = vectorstore.similarity_search("Word2Vec")
# Print the page_content property of the first element in the sub_docs list
print(sub_docs[0].page_content)

Word2Vec


In [43]:
# Retrieve and fetch documents
retrieved_docs = retriever.invoke("Word2Vec")

# Return the length of the page content of the first retrieved document
print(retrieved_docs[0].page_content)

Digital Transformation

Definition: Digital transformation refers to the process of leveraging technology to innovate a company’s services, culture, and operations, enhancing competitiveness through digital solutions.
Example: A company adopting cloud computing to revolutionize its data storage and processing is an example of digital transformation.
Related Keywords: Innovation, Technology, Business Model

Crawling

Definition: Crawling is the automated process of visiting web pages to collect data. It is commonly used in search engine optimization and data analysis.
Example: Google’s search engine crawls websites to collect content and index it.
Related Keywords: Data Collection, Web Scraping, Search Engine

Word2Vec


## MultiQueryRetriever

In [2]:
from langchain_community.document_loaders import WebBaseLoader
from langchain.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Load a blog post
loader = WebBaseLoader(
    "https://python.langchain.com/docs/introduction/", encoding="utf-8"
)

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=0)
docs = loader.load_and_split(text_splitter)

openai_embeddings = OpenAIEmbeddings()

db = FAISS.from_documents(docs, openai_embeddings)

retriever = db.as_retriever()

query = "Please explain the key features and architecture of the LangChain framework."
relevant_docs = retriever.invoke(query)

# Print the number of retrieved documents
print(f"Number of retrieved documents: {len(relevant_docs)}")

for idx, doc in enumerate(relevant_docs, start=1):
    print(f"Document #{idx}:\n{doc.page_content}\n{'-'*40}")

USER_AGENT environment variable not set, consider setting it to identify your requests.


Number of retrieved documents: 4
Document #1:
LangChain is a framework for developing applications powered by large language models (LLMs).
LangChain simplifies every stage of the LLM application lifecycle:
----------------------------------------
Document #2:
model.invoke("Hello, world!")
noteThese docs focus on the Python LangChain library. Head here for docs on the JavaScript LangChain library.
Architecture​
The LangChain framework consists of multiple open-source libraries. Read more in the
Architecture page.
----------------------------------------
Document #3:
However, these guides will help you quickly accomplish common tasks using chat models,
vector stores, and other common LangChain components.
Check out LangGraph-specific how-tos here.
Conceptual guide​
Introductions to all the key parts of LangChain you’ll need to know! Here you'll find high level explanations of all LangChain concepts.
For a deeper dive into LangGraph concepts, check out this page.
Integrations​
----------

In [3]:
from langchain.retrievers.multi_query import MultiQueryRetriever
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini")

multiquery_retriever = MultiQueryRetriever.from_llm(
    retriever=db.as_retriever(),
    llm=llm
)

In [4]:
# Logging settings for the query
import logging

logging.basicConfig()
logging.getLogger("langchain.retrievers.multi_query").setLevel(logging.INFO)

In [5]:
question =  "Please explain the key features and architecture of the LangChain framework."

relavant_docs = multiquery_retriever.invoke(question)

# Return the number of unique documents retrieved.
print(
    f"===============\nNumber of retrieved documents: {len(relevant_docs)}",
    end="\n===============\n",
)

# Print the content of the retrieved documents.
print(relevant_docs[0].page_content)

INFO:langchain.retrievers.multi_query:Generated queries: ['What are the main components and design principles of the LangChain framework?', 'Can you provide an overview of the architecture and essential features of LangChain?', 'How does the LangChain framework work, and what are its core functionalities?']


Number of retrieved documents: 4
LangChain is a framework for developing applications powered by large language models (LLMs).
LangChain simplifies every stage of the LLM application lifecycle:


In [6]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Define the prompt template (written to generate 5 questions)
prompt = PromptTemplate.from_template(
    """You are an AI language model assistant. 
Your task is to generate five different versions of the given user question to retrieve relevant documents from a vector database. 
By generating multiple perspectives on the user question, your goal is to help the user overcome some of the limitations of the distance-based similarity search. 
Your response should be a list of values separated by new lines, eg: `foo\nbar\nbaz\n`

#ORIGINAL QUESTION: 
{question}

#Answer in English:
"""
)

custom_multiquery_chain = (
    {"question": RunnablePassthrough()} | prompt | llm | StrOutputParser()
)

# Define the question.
question = "Please explain the key features and architecture of the LangChain framework."

# Execute the chain and check the generated multiple queries.
multi_queries = custom_multiquery_chain.invoke(question)
# Check the result (5 generated questions)
print(multi_queries)

What are the main components and structure of the LangChain framework?  
Can you describe the architecture and essential characteristics of LangChain?  
What key features define the LangChain framework and how is it architected?  
Could you elaborate on the architectural design and significant features of LangChain?  
What are the crucial aspects and the framework layout of LangChain?  


In [7]:
multiquery_retriever = MultiQueryRetriever.from_llm(
    llm=custom_multiquery_chain,
    retriever=db.as_retriever()
)

In [8]:
# Result
relevant_docs = multiquery_retriever.invoke(question)

# Return the number of unique documents retrieved.
print(
    f"===============\nNumber of retrieved documents: {len(relevant_docs)}",
    end="\n===============\n",
)

# Print the content of the retrieved documents.
print(relevant_docs[0].page_content)

INFO:langchain.retrievers.multi_query:Generated queries: ['Could you outline the main features and architectural components of the LangChain framework?  ', 'What are the important characteristics and structure of the LangChain framework?  ', 'Can you describe the key aspects and design of the LangChain framework?  ', 'What does the LangChain framework consist of in terms of its features and architecture?  ', 'Can you provide an overview of the primary features and the architecture underlying the LangChain framework?']


Number of retrieved documents: 6
LangChain is a framework for developing applications powered by large language models (LLMs).
LangChain simplifies every stage of the LLM application lifecycle:


## MultiVector

In [9]:
from langchain_community.document_loaders import PyMuPDFLoader

loader = PyMuPDFLoader("A European Approach to Artificial Intelligence - A Policy Perspective.pdf")
docs = loader.load()

In [10]:
print(docs[5].page_content[:500])

A EUROPEAN APPROACH TO ARTIFICIAL INTELLIGENCE - A POLICY PERSPECTIVE
6
data for innovators, particularly in the business-to-business (B2B) 
or government-to-citizens (G2C) domains: e.g. by open access to 
government data in sectors such as transportation and health-
care (Burghin et al., 2019), privacy-preserving data marketplaces 
for companies to share data (de Streel et al., 2019). The genuine 
concern for innovators access to data is shown by the city of Bar-
celona where ‘data sovereignty’


In [11]:
import uuid
from langchain.storage import InMemoryStore
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.retrievers.multi_vector import MultiVectorRetriever

vectorstore = Chroma(collection_name="small_bigger_chunks", embedding_function=OpenAIEmbeddings(model="text-embedding-3-small"))

store = InMemoryStore()

id_key = "doc_id"

retriever = MultiVectorRetriever(
    vectorstore=vectorstore,
    byte_store=store,
    id_key=id_key
)

doc_ids = [str(uuid.uuid4()) for _ in docs]

print(doc_ids[:2])

['39a3e9eb-9053-4616-944a-e6556bc971ca', '8f1f5cf1-3ceb-43d4-8246-1449be76e157']


In [12]:
# Create a RecursiveCharacterTextSplitter object for larger chunks
parent_text_splitter = RecursiveCharacterTextSplitter(chunk_size=600)

# Splitter to be used for generating smaller chunks
child_text_splitter = RecursiveCharacterTextSplitter(chunk_size=200)

In [13]:
parent_docs = []

for i, doc in enumerate(docs):
    _id = doc_ids[i]
    parent_doc = parent_text_splitter.split_documents([doc])

    for _doc in parent_doc:
        _doc.metadata[id_key] = _id
    
    parent_docs.extend(parent_doc)

In [14]:
parent_docs[0].metadata

{'producer': 'Adobe PDF Library 15.0',
 'creator': 'Adobe InDesign 15.1 (Macintosh)',
 'creationdate': '2020-09-22T22:35:34+02:00',
 'source': 'A European Approach to Artificial Intelligence - A Policy Perspective.pdf',
 'file_path': 'A European Approach to Artificial Intelligence - A Policy Perspective.pdf',
 'total_pages': 24,
 'format': 'PDF 1.4',
 'title': '',
 'author': '',
 'subject': '',
 'keywords': '',
 'moddate': '2020-09-22T22:35:44+02:00',
 'trapped': '',
 'modDate': "D:20200922223544+02'00'",
 'creationDate': "D:20200922223534+02'00'",
 'page': 0,
 'doc_id': '39a3e9eb-9053-4616-944a-e6556bc971ca'}

In [15]:
child_docs = []
for i, doc in enumerate(docs):
    # Retrieve the ID of the current document
    _id = doc_ids[i]
    # Split the current document into child documents
    child_doc = child_text_splitter.split_documents([doc])
    for _doc in child_doc:
        # Set the document ID in the metadata
        _doc.metadata[id_key] = _id
    child_docs.extend(child_doc)

In [16]:
# Check the metadata of the generated child documents.
child_docs[0].metadata

{'producer': 'Adobe PDF Library 15.0',
 'creator': 'Adobe InDesign 15.1 (Macintosh)',
 'creationdate': '2020-09-22T22:35:34+02:00',
 'source': 'A European Approach to Artificial Intelligence - A Policy Perspective.pdf',
 'file_path': 'A European Approach to Artificial Intelligence - A Policy Perspective.pdf',
 'total_pages': 24,
 'format': 'PDF 1.4',
 'title': '',
 'author': '',
 'subject': '',
 'keywords': '',
 'moddate': '2020-09-22T22:35:44+02:00',
 'trapped': '',
 'modDate': "D:20200922223544+02'00'",
 'creationDate': "D:20200922223534+02'00'",
 'page': 0,
 'doc_id': '39a3e9eb-9053-4616-944a-e6556bc971ca'}

In [17]:
print(f"Number of split parent_docs: {len(parent_docs)}")
print(f"Number of split child_docs: {len(child_docs)}")

Number of split parent_docs: 177
Number of split child_docs: 950


In [18]:
retriever.vectorstore.add_documents(parent_docs)
retriever.vectorstore.add_documents(child_docs)

retriever.docstore.mset(list(zip(doc_ids, docs)))

In [19]:
# Perform similarity search on the vectorstore
relevant_chunks = retriever.vectorstore.similarity_search(
    "What is the phased implementation timeline for the EU AI Act?"
)
print(f"Number of retrieved documents: {len(relevant_chunks)}")

Number of retrieved documents: 4


In [20]:
for chunk in relevant_chunks:
    print(chunk.page_content, end="\n\n")
    print(">" * 100, end="\n\n")

peration on AI (European Commission, 2018c), and coordinated 
action plan on the development of AI in the EU (European Com-
mission, 2018d), among others. The European strategy aims to

>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>

Europe’ (European Commission, 2018a), the declaration of coo-
peration on AI (European Commission, 2018c), and coordinated 
action plan on the development of AI in the EU (European Com-

>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>

A EUROPEAN APPROACH TO ARTIFICIAL INTELLIGENCE - A POLICY PERSPECTIVE
10
requirements becomes mandatory in all sectors and create bar-
riers especially for innovators and SMEs. Public procurement ‘data 
sovereignty clauses’ induce large players to withdraw from AI for 
urban ecosystems. Strict liability sanctions block AI in healthcare, 
while limiting space of self-driving experimentation. The support 
measures to boos

In [21]:
relevant_docs = retriever.invoke(
    "What is the phased implementation timeline for the EU AI Act?"
)
print(f"Number of retrieved documents: {len(relevant_docs)}", end="\n\n")
print("=" * 100, end="\n\n")
print(relevant_docs[0].page_content)

Number of retrieved documents: 2


A EUROPEAN APPROACH TO ARTIFICIAL INTELLIGENCE - A POLICY PERSPECTIVE
5
laws and regulation. Some negative examples have been given 
wide attention in the media: a fatal accident involving an autono-
mous vehicle2; Microsoft’s chatting bot Tay being shut down after 
16 hours because it became racist, sexist, and denied the Holo-
caust3; racially biased decisions with credit checks and recidivism 
(Teich & Tirias Research, 2018). Such examples are fuelling a va-
riety of concerns about accountability, fairness, bias, autonomy, 
and due process of AI systems (Pasquale, 2015; Ziewitz, 2015). 
Beyond these anecdotal instances, AI presents several challenges 
(Dwivedi et al., 2019), which are economic (need of funds, impact 
on employment and performances) and organizational (changing 
working practices, cultural barriers, need of new skills, data inte-
gration, etc.) issues to be tackled. At societal level AI may challenge 
cultural norms and face resista

In [22]:
from langchain.retrievers.multi_vector import SearchType

# Set the search type to Maximal Marginal Relevance (MMR)
retriever.search_type = SearchType.mmr

# Search all related documents
print(
    retriever.invoke(
        "What is the phased implementation timeline for the EU AI Act?"
    )[0].page_content
)

A EUROPEAN APPROACH TO ARTIFICIAL INTELLIGENCE - A POLICY PERSPECTIVE
5
laws and regulation. Some negative examples have been given 
wide attention in the media: a fatal accident involving an autono-
mous vehicle2; Microsoft’s chatting bot Tay being shut down after 
16 hours because it became racist, sexist, and denied the Holo-
caust3; racially biased decisions with credit checks and recidivism 
(Teich & Tirias Research, 2018). Such examples are fuelling a va-
riety of concerns about accountability, fairness, bias, autonomy, 
and due process of AI systems (Pasquale, 2015; Ziewitz, 2015). 
Beyond these anecdotal instances, AI presents several challenges 
(Dwivedi et al., 2019), which are economic (need of funds, impact 
on employment and performances) and organizational (changing 
working practices, cultural barriers, need of new skills, data inte-
gration, etc.) issues to be tackled. At societal level AI may challenge 
cultural norms and face resistance (Hu et al, 2019). In Europe the

In [23]:
# Importing libraries for loading PDF files and splitting text
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Initialize the PDF file loader
loader = PyMuPDFLoader("A European Approach to Artificial Intelligence - A Policy Perspective.pdf")

# Split text
text_splitter = RecursiveCharacterTextSplitter(chunk_size=600, chunk_overlap=50)

# Load a PDF file and run Text Split
split_docs = loader.load_and_split(text_splitter)

# Output the number of split documents
print(f"Number of split documents: {len(split_docs)}")

Number of split documents: 135


In [24]:
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

summary_chain = (
    {"doc": lambda x: x.page_content}
    # Create a prompt template for document summaries
    | ChatPromptTemplate.from_messages(
        [
            ("system", "You are an expert in summarizing documents in English."),
            (
                "user",
                "Summarize the following documents in 3 sentences in bullet points format.\n\n{doc}",
            ),
        ]
    )
    # Using OpenAI's ChatGPT model to generate summaries
    | ChatOpenAI(temperature=0, model="gpt-4o-mini")
    | StrOutputParser()
)

In [25]:
summaries = summary_chain.batch(split_docs, {"max_concurrency": 10})

In [26]:
len(summaries)

135

In [27]:
# Prints the contents of the original document.
print(split_docs[33].page_content, end="\n\n")
# Print a summary.
print("[summary]")
print(summaries[33])

decision-making process may become less tractable9. The chosen 
decision model may also turn out to be unsuitable if the real-world 
environment behaves differently from what was expected. While 
more and better data be used for training can help improving pre-
diction, it will never be perfect or include all justifiable outliers. On 
the other hand, as technology advances more instruments may 
become available to quantify the degree of influence of input va-
riables on algorithm outputs (Datta et al., 2016). Research is also 
underway in pursuit of rendering algorithms more amenable to

[summary]
- The decision-making process can become complicated if the chosen model does not align with real-world conditions, leading to potential unsuitability.  
- Although improved data can enhance predictions, it will never be flawless or encompass all relevant outliers.  
- Advancements in technology may provide new tools to measure the impact of input variables on algorithm outputs, and research 

In [28]:
import uuid

# Create a vector store to store the summary information.
summary_vectorstore = Chroma(
    collection_name="summaries",
    embedding_function=OpenAIEmbeddings(model="text-embedding-3-small"),
)

# Create a repository to store the parent document.
store = InMemoryStore()

# Specify a key name to store the document ID.
id_key = "doc_id"

# Initialize the searcher (empty at startup).
retriever = MultiVectorRetriever(
    vectorstore=summary_vectorstore,  # vector store
    byte_store=store,  # byte store
    id_key=id_key,  # document ID
)
# Create a document ID.
doc_ids = [str(uuid.uuid4()) for _ in split_docs]

In [29]:
summary_docs = [
    # Create a Document object with the summary as the page content and the document ID as metadata.
    Document(page_content=s, metadata={id_key: doc_ids[i]})
    for i, s in enumerate(summaries)
]

In [30]:
# Number of documents in the summary
len(summary_docs)

135

In [31]:
retriever.vectorstore.add_documents(summary_docs)

retriever.docstore.mset(list(zip(doc_ids, split_docs)))

In [32]:
# Perform a similarity search.
result_docs = summary_vectorstore.similarity_search(
    "What is the phased implementation timeline for the EU AI Act?"
)

In [33]:
# Output 1 result document.
print(result_docs[0].page_content)

- The European Commission and EU member states are collaborating to enhance the development and implementation of artificial intelligence (AI) technologies within Europe.  
- A White Paper published in 2020 outlines a European approach to AI, emphasizing the importance of excellence and trust in AI systems.  
- The initiative aims to position Europe as a leader in AI innovation while ensuring ethical standards and regulatory frameworks are established.  


In [34]:
# Search for and fetch related articles.
retrieved_docs = retriever.invoke(
    "What is the phased implementation timeline for the EU AI Act?"
)
print(retrieved_docs[0].page_content)

cial Intelligence. Retrieved from https://ec.europa.eu/digital-single-market/en/news/
eu-member-states-sign-cooperate-artificial-intelligence.
European Commission. (2018d). Member States and Commission to work together to 
boost artificial intelligence ‘made in Europe’. Retrieved from https://ec.europa.eu/commis-
sion/presscorner/detail/en/IP_18_6689.
European Commission. (2020a). White Paper on Artificial Intelligence. A European Ap-
proach to Excellence and Trust. COM(2020) 65 final, Brussels: European Commission.


In [35]:
functions = [
    {
        "name": "hypothetical_questions",  # Specify a name for the function.
        "description": "Generate hypothetical questions",  # Write a description of the function.
        "parameters": {  # Define the parameters of the function.
            "type": "object",  # Specifies the type of the parameter as an object.
            "properties": {  # Defines the properties of an object.
                "questions": {  # Define the 'questions' attribute.
                    "type": "array",  # Type 'questions' as an array.
                    "items": {
                        "type": "string"
                    },  # Specifies the array's element type as String.
                },
            },
            "required": ["questions"],  # Specify 'questions' as a required parameter.
        },
    }
]

In [36]:
from langchain_core.prompts import ChatPromptTemplate
from langchain.output_parsers.openai_functions import JsonKeyOutputFunctionsParser
from langchain_openai import ChatOpenAI

hypothetical_query_chain = (
    {"doc": lambda x: x.page_content}
    # We ask you to create exactly 3 hypothetical questions that you can answer using the documentation below. This number can be adjusted.
    | ChatPromptTemplate.from_template(
        "Generate a list of exactly 3 hypothetical questions that the below document could be used to answer. "
        "Potential users are those interested in the AI industry. Create questions that they would be interested in. "
        "Output should be written in English:\n\n{doc}"
    )
    | ChatOpenAI(max_retries=0, model="gpt-4o-mini").bind(
        functions=functions, function_call={"name": "hypothetical_questions"}
    )
    # Extract the value corresponding to the “questions” key from the output.
    | JsonKeyOutputFunctionsParser(key_name="questions")
)

In [37]:
hypothetical_query_chain.invoke(split_docs[33])

['How might the decision-making process in AI evolve if researchers find ways to better quantify the influence of input variables on algorithm outputs?',
 'What could be the implications for AI industries if decision models consistently prove to be unsuitable due to unexpected real-world behaviors?',
 'If the accuracy of AI predictions improves significantly with access to better data, how might that change the landscape of decision-making in businesses relying on AI technologies?']

In [38]:
# Create a batch of hypothetical questions for a list of articles
hypothetical_questions = hypothetical_query_chain.batch(
    split_docs, {"max_concurrency": 10}
)

In [39]:
hypothetical_questions[33]

['How might decision models need to evolve if real-world environments continue to diverge from expected behavior due to rapid technological advancements in AI?',
 'What could be the potential consequences of relying on algorithms that are trained with imperfect data in critical decision-making scenarios?',
 'In what ways could emerging technologies improve our understanding of how input variables influence algorithm outputs, and what implications might this have for the AI industry?']

In [40]:
# Vector store to use for indexing child chunks
hypothetical_vectorstore = Chroma(
    collection_name="hypo-questions", embedding_function=OpenAIEmbeddings()
)
# Storage hierarchy for parent documents
store = InMemoryStore()

id_key = "doc_id"
# Retriever (empty on startup)
retriever = MultiVectorRetriever(
    vectorstore=hypothetical_vectorstore,
    byte_store=store,
    id_key=id_key,
)
doc_ids = [str(uuid.uuid4()) for _ in split_docs]  # Create a document ID

In [41]:
question_docs = []
# save hypothetical_questions
for i, question_list in enumerate(hypothetical_questions):
    question_docs.extend(
        # Create a Document object for each question in the list of questions, and include the document ID for that question in the metadata.
        [Document(page_content=s, metadata={id_key: doc_ids[i]}) for s in question_list]
    )

In [42]:
# Add the hypothetical_questions document to the vector repository.
retriever.vectorstore.add_documents(question_docs)

# Map the document ID to the document and store it in the document store.
retriever.docstore.mset(list(zip(doc_ids, split_docs)))

In [43]:
# Search the vector repository for similar documents.
result_docs = hypothetical_vectorstore.similarity_search(
    "What is the phased implementation timeline for the EU AI Act?"
)

In [44]:
# Output the results of the similarity search.
for doc in result_docs:
    print(doc.page_content)
    print(doc.metadata)

What potential socio-economic changes could arise from the implementation of the EU's coordinated action plan on AI?
{'doc_id': 'b003e309-bdb6-47b6-a89b-03ac95283d77'}
How would the implementation of the European AI policy shape future innovations in the AI industry?
{'doc_id': 'd4e8debb-9c93-4fd9-98ce-5f5233f330f0'}
How might the implementation of stricter regulations in the EU impact the development and deployment of AI technologies in Europe?
{'doc_id': '85fec8e8-199e-4cf5-84be-ab4d6697aeb8'}
If industry leaders were to implement the recommendations from this report, what potential changes could we see in the European AI landscape over the next five years?
{'doc_id': '577370dd-d80c-4767-89a7-e13cbc8c2f13'}


In [45]:
# Search for and fetch related articles.
retrieved_docs = retriever.invoke(result_docs[1].page_content)

# Output the documents found.
for doc in retrieved_docs:
    print(doc.page_content)

A EUROPEAN APPROACH TO ARTIFICIAL INTELLIGENCE - A POLICY PERSPECTIVE
20
Figure 4: Scenarios assessment,
Based on the considerations in the previous sections, a qualitative 
assessment of the four scenarios with regard to four dimensions 
(Growth, Innovation, Fairness, and Trust), representing high-level 
policy objectives, was conducted. The scenarios have been scored 
in the strict order from least (1) to most impact (4) with regard 
to the four dimensions of assessment, thus providing a relative 
comparison between the scenarios. This has been depicted in the
A EUROPEAN APPROACH TO ARTIFICIAL INTELLIGENCE - A POLICY PERSPECTIVE
24
Publisher
EIT Digital
Rue Guimard 7
1040 Brussels
Belgium
www.eitdigital.eu
Contact
info@eitdigital.eu
ISBN 978-91-87253-64-5
reof, of algorithms on which AI applications rely. There is a need 
to study and understand where algorithms may go wrong as to 
adopt adequate and proportional remedial and mitigation mea-
sures. Algorithmic rules may imply moral j